In [1]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
sys.path.append(r"../../NeuralNetworks/Utils")
import TCP_IP_UTILS
import NN_Utils
import Utils
import time
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import struct
from torch.cuda.amp import autocast

In [2]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

Available VISA resources:
 - USB0::0x0957::0x1F01::MY53270560::0::INSTR
 - USB0::0x0957::0x1F01::MY57280340::0::INSTR
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9007::MY62310119::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12


In [3]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Default Signals Set
[128]


In [4]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.02)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

['+2.99775900E+00', '+5.49300000E-03']
Default Signals Set
[128]


In [5]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.02)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

['+2.99735100E+00', '+1.29960000E-02']


In [6]:
#Turn on CLK 
#Device_ID_CLK = "USB0::0x0957::0x1F01::MY57280340::0::INSTR"
Device_ID_CLK = "USB0::0x0957::0x1F01::MY53270560::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

Connected to: Agilent Technologies, N5173B, MY53270560, B.01.70


In [7]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_LM["id"]
scan_len = Utils.DL_EN_SC_LM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[4] = 0
#scan_data[2] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [8]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_RM["id"]
scan_len = Utils.DL_EN_SC_RM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[71] = 0
#scan_data[4] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [9]:
#Control Logic TD Data
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b0;
    ENTIME_EXT_LOC = 1'b1;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b0;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b0;

    IN_LB = 6'd3;
    IN_UB = 6'd22; 
    TDC_EN_LB = 6'd1;
    TDC_EN_UB = 6'd21;
    ENCHG_LB = 6'd0;
    ENCHG_UB = 6'd0;
    ENTIME_LB = 6'd0;
    ENTIME_UB = 6'd0;
    PCHG_LB = 6'd2;
    PCHG_UB = 6'd22;
    VDAC_CTRL_LB = 6'd0;
    VDAC_CTRL_UB = 6'd0; 
    DEL_RST_LB = 6'd2;
    DEL_RST_UB = 6'd22;
    TDC_COMPUTE_LB = 6'd19;
    TDC_COMPUTE_UB = 6'd24;
    TDC_RST_LB = 6'd1;
    TDC_RST_UB = 6'd21;
    BL_NUM_TGT = 2'd3;
    BL_DONE_TIME = 6'd55;
    VTC_EN_LB = 6'd0;
    VTC_EN_UB = 6'd0;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd0;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [10]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = scan_gen(5)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
#print(ret_data)

000101
[162, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received


In [11]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [12]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.04)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

['+2.99815000E+00', '+2.48910000E-02']


In [13]:
flattened = (Utils.TD_CORRECTION).flatten()
ret_data = Utils.Send_CorrectionData(client_socket,flattened,0)
print(ret_data)

[128]


In [14]:
# 🧩 Step 2: Load Raw IDX Files (not .gz)
def load_images_idx3(path):
    with open(path, 'rb') as f:
        _ = struct.unpack(">I", f.read(4))  # magic number
        num = struct.unpack(">I", f.read(4))[0]
        rows = struct.unpack(">I", f.read(4))[0]
        cols = struct.unpack(">I", f.read(4))[0]
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(num, 1, rows, cols)

def load_labels_idx1(path):
    with open(path, 'rb') as f:
        _ = struct.unpack(">I", f.read(4))  # magic number
        num = struct.unpack(">I", f.read(4))[0]
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data
# 🧩 Step 3: Load Dataset
X_test = load_images_idx3(r"../../NeuralNetworks/Dataset/MNIST/test/t10k-images.idx3-ubyte")
y_test = load_labels_idx1(r"../../NeuralNetworks/Dataset/MNIST/test/t10k-labels.idx1-ubyte")

# X_train = load_images_idx3(r"../Dataset/MNIST/train/train-images.idx3-ubyte")
# y_train = load_labels_idx1(r"../Dataset/MNIST/train/train-labels.idx1-ubyte")
# 🧩 Step 4: Wrap in PyTorch Dataset
class MNISTDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32) / 255.0
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# train_loader = DataLoader(MNISTDataset(X_train, y_train), batch_size=64, shuffle=True)
test_dataset = MNISTDataset(X_test, y_test)
test_loader = DataLoader(MNISTDataset(X_test, y_test), batch_size=1, shuffle=False)




In [15]:
# Define CNN model
class SimpleCNN(nn.Module):
    def __init__(self, use_custom=[False,False,False,False],precision = [],HW_Modelled = [], client_socket = None, LUT_FLAGS = [], LUT_PATHS = []):
        super(SimpleCNN, self).__init__()
        self.use_custom = use_custom
        self.precision = precision
        self.HW_Modelled = HW_Modelled
        self.client_socket = client_socket
        self.LUT_FLAGS = LUT_FLAGS
        self.LUT_PATHS = LUT_PATHS
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)   # (28x28) -> (26x26)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)  # (26x26) -> (24x24)
        self.pool = nn.MaxPool2d(2, 2)                 # (24x24) -> (12x12)
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        sum = None
        if self.use_custom[0]:
            custom_conv0 = NN_Utils.Convolution(self.HW_Modelled[0],x,self.conv1.weight,self.precision[0],self.conv1.stride[0],self.conv1.padding[0],self.conv1.bias,self.client_socket,self.LUT_FLAGS[0],self.LUT_PATHS[0])
            x,sum = custom_conv0.convolution_onChip()
            x = torch.relu(x)
        else:
            x = torch.relu(self.conv1(x))        # -> (32, 26, 26)
            
        if self.use_custom[1]:
            custom_conv1 = NN_Utils.Convolution(self.HW_Modelled[1],x,self.conv2.weight,self.precision[1],self.conv2.stride[0],self.conv2.padding[0],self.conv2.bias,self.client_socket,self.LUT_FLAGS[1],self.LUT_PATHS[1])
            x,sum = custom_conv1.convolution_onChip()
            x = torch.relu(x)
        else:    
            x = torch.relu(self.conv2(x))        # -> (64, 24, 24)

        x = self.pool(x)                 # -> (64, 12, 12)
        x = x.view(-1, 64 * 12 * 12)     # flatten

        if self.use_custom[2]:
            lin1 = NN_Utils.Linear(self.HW_Modelled[2],x,self.fc1.weight,self.precision[2],self.fc1.bias,self.client_socket,self.LUT_FLAGS[2],self.LUT_PATHS[2])
            x,sum = lin1.linear_onChip()
            x = torch.relu(x)
        else:
            x = torch.relu(self.fc1(x))
        if self.use_custom[3]:
            lin2 = NN_Utils.Linear(self.HW_Modelled[3],x,self.fc2.weight,self.precision[3],self.fc2.bias,self.client_socket,self.LUT_FLAGS[3],self.LUT_PATHS[3])
            x,sum = lin2.linear_onChip()
        else:
            x = self.fc2(x)
        return x,sum
    

In [16]:
from matplotlib import pyplot as plt
def show_images(imgs, preds, labels, max_images=4):
    """
    imgs: Tensor [B, C, H, W]
    preds: Tensor [B]
    labels: Tensor [B]
    """
    imgs = imgs.cpu()
    preds = preds.cpu()
    labels = labels.cpu()

    batch_size = min(imgs.size(0), max_images)

    plt.figure(figsize=(12, 3))
    for i in range(batch_size):
        img = imgs[i]

        # If normalized, uncomment and adjust these values
        # mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        # std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        # img = img * std + mean

        img = img.permute(1, 2, 0)  # C,H,W → H,W,C
        img = img.clamp(0, 1)

        plt.subplot(1, batch_size, i + 1)
        plt.imshow(img)
        plt.title(f"Pred: {preds[i].item()}\nActual: {labels[i].item()}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [17]:

TD_DEL_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_TD_DEL.npy"
TD_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_TD_FastIMC.npy" #r"../Utils/Hardware_Model/HW_MODEL_TD_1x_Ref256.npy"
CD_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_CD_FastIMC.npy" #r"../Utils/Hardware_Model/HW_MODEL_CD_4x_Ref256_sweep4.npy"
LUT_PATHS = [[TD_DEL_LUT_PATH,CD_LUT_PATH],[TD_LUT_PATH,CD_LUT_PATH],[TD_LUT_PATH,CD_LUT_PATH],[TD_LUT_PATH,CD_LUT_PATH]]
LUT_FLAGS = [[True,False,True],[False,True,True],[False,True,True],[False,True,True]]

In [18]:
correct = 0
total = 0
count = 0


In [19]:
# 🧠 Load model from file

for layer in [1]:
    custom = [False,False,False,True]
    #custom[layer] = True
    for pr in [[7,7]]:
        precision = [[0,0],[0,0],[0,0],[0,0]]
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = SimpleCNN(HW_Modelled=[1,1,1,0],use_custom=custom,precision=precision,client_socket=client_socket,LUT_PATHS=LUT_PATHS,LUT_FLAGS=LUT_FLAGS).to(device)
        model.load_state_dict(torch.load(r"../../NeuralNetworks/Model_Zoo/MNIST/simple_mnist_cnn_half.pth", map_location=device))
        model.eval()
        # ✅ Run inference and calculate accuracy98
        #correct = 0
        #total = 0
        #count = 0
        partial_sum = torch.tensor([], dtype=torch.uint8, device=device)
        partial_sum2 = torch.tensor([], dtype=torch.uint8, device=device)
        with torch.no_grad():
            #for imgs, labels in list(test_loader):
            while(1):
                imgs, labels = list(test_loader)[count]
                imgs, labels = imgs.to(device), labels.to(device)
                #with autocast():
                outputs,sum = model(imgs)
                #if sum is not None:
                    #partial_sum = torch.cat((partial_sum,sum),dim=0)
                    #partial_sum2 = torch.cat((partial_sum2,sum2),dim=0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                count +=1
                #show_images(imgs, preds, labels, max_images=4)
                print(f"Batches Done : {count} Correct : {correct} Total : {total} Inference Accuracy: {100 * correct / total:.2f}%\n\n") 
                if count == 20: break
        torch.cuda.empty_cache()
        print(f"Batches Done : {count} Correct : {correct} Total : {total} Final Inference Accuracy: {100 * correct / total:.2f}%\n\n")

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 7
Received Data Size: 256
All Data Received
Batches Done : 1 Correct : 1 Total : 1 Inference Accuracy: 100.00%


Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 7
Received Data Size: 256
All Data Received
Batches Done : 2 Correct : 2 Total : 2 Inference Accuracy: 100.00%


Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 7
Received Data Size: 256
All Data Received
Batches Done : 3 Correct : 3 Total : 3 Inference Accuracy: 100.00%


Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 7
Received Data Size: 256
All Data Received
Batches Done : 4 Correct : 4 Total : 4 Inference Accuracy: 100.00%


Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 7
Received Data Size: 256
All Data Received
Batches Done : 5 Correct : 5 Total : 5 Inference Accuracy: 100.00%


Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 7
Received Data Size: 256
All Data Received
B

In [23]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 5
Received Data Size: 46240
All Data Received
46240
[85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [24]:
unique = set(SRAM_DATA)
print(unique)

{0, 85}


In [22]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [25]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [26]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0